# Medical QLoRA Fine-Tuning with Unsloth (Colab)

This notebook implements an end-to-end PEFT workflow:
- 4-bit quantized base model loading
- LoRA adapter setup via Unsloth
- Medical dataset formatting + tokenization
- Epoch-based training
- GPU memory/performance monitoring
- Adapter save + inference tests

In [ ]:
# ---- 1) Install dependencies (Colab) ----
# If this is your first run in a new runtime, execute this cell first.

%%capture
!pip -q install --upgrade pip
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install -U datasets trl accelerate peft transformers bitsandbytes

print("Dependencies installed.")

In [ ]:
# ---- 2) Imports and environment check ----
import os
import time
import torch

from datasets import load_dataset
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported

assert torch.cuda.is_available(), "GPU is required. In Colab: Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"Total VRAM: {total_gb:.2f} GB")

In [ ]:
# ---- 3) Optional: mount Google Drive for persistent outputs ----
USE_DRIVE = False  # set True if you want to save adapter to Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/medical_qlora_adapter"

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f"Drive output dir ready: {DRIVE_OUTPUT_DIR}")
else:
    print("Drive mount skipped.")

In [ ]:
# ---- 4) Training + generation configuration ----
BASE_MODEL = "unsloth/DeepSeek-R1-Distill-Llama-8B-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True

# Dataset config (this notebook is wired for medmcqa)
DATASET_NAME = "medmcqa"
MAX_TRAIN_SAMPLES = 5000    # increase for better quality if time allows

# Training hyperparameters
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
MAX_STEPS = -1              # set > 0 to cap training steps
BATCH_SIZE = 2
GRAD_ACCUM = 4
WARMUP_STEPS = 50
LOGGING_STEPS = 10

# Generation config for inference
GEN_MAX_NEW_TOKENS = 220
GEN_TEMPERATURE = 0.3
GEN_TOP_P = 0.9

OUTPUT_DIR = "/content/medical_qlora_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config loaded.")

In [ ]:
# ---- 5) Load base model (4-bit) + tokenizer ----
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("Model + LoRA adapters ready.")

In [ ]:
# ---- 6) Load and format medical dataset ----
# medmcqa fields include: question, opa, opb, opc, opd, cop (correct option index), subject_name, topic_name
raw_train = load_dataset(DATASET_NAME, split="train")

print("Raw medmcqa train split:")
print(raw_train)
print("First sample (raw):")
print({k: raw_train[0][k] for k in ['question', 'opa', 'opb', 'opc', 'opd', 'cop'] if k in raw_train.column_names})
print()

if MAX_TRAIN_SAMPLES > 0:
    raw_train = raw_train.shuffle(seed=42).select(range(min(MAX_TRAIN_SAMPLES, len(raw_train))))
    print(f"Subsampled to {len(raw_train)} training examples.")

alpaca_prompt = (
    "Below is a medical question. Provide a clinically sound answer.\n\n"
    "### Question:\n{}\n\n"
    "### Context:\n{}\n\n"
    "### Answer:\n{}"
 )

letter_map = {0: "A", 1: "B", 2: "C", 3: "D"}

def format_medmcqa(example):
    question = example.get("question", "").strip()
    options = [
        f"A) {example.get('opa', '')}",
        f"B) {example.get('opb', '')}",
        f"C) {example.get('opc', '')}",
        f"D) {example.get('opd', '')}",
    ]
    context = (
        f"Subject: {example.get('subject_name', 'Unknown')}\n"
        f"Topic: {example.get('topic_name', 'Unknown')}\n"
        + "\n".join(options)
    )
    idx = example.get("cop", -1)
    correct_letter = letter_map.get(idx, "Unknown")
    answer_text = ""
    if idx == 0:
        answer_text = example.get('opa', '')
    elif idx == 1:
        answer_text = example.get('opb', '')
    elif idx == 2:
        answer_text = example.get('opc', '')
    elif idx == 3:
        answer_text = example.get('opd', '')

    answer = f"Correct option: {correct_letter}\nRationale: {answer_text}"

    text = alpaca_prompt.format(question, context, answer) + tokenizer.eos_token
    return {"text": text}

train_dataset = raw_train.map(format_medmcqa, remove_columns=raw_train.column_names)
print(train_dataset)
print("Sample formatted training text (truncated):\n")
print(train_dataset[0]["text"][:800])

In [ ]:
# ---- 7) Memory monitor callback ----
class GpuMemoryCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            peak = torch.cuda.max_memory_allocated() / 1024**3
            print(f"[GPU] alloc={allocated:.2f} GB | reserved={reserved:.2f} GB | peak={peak:.2f} GB")

print("Memory callback ready.")

In [ ]:
# ---- 8) Trainer setup + epoch-based fine-tuning ----
bf16 = is_bfloat16_supported()

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=WARMUP_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not bf16,
    bf16=bf16,
    logging_steps=LOGGING_STEPS,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir=OUTPUT_DIR,
    report_to="none",
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    save_strategy="epoch",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
    callbacks=[GpuMemoryCallback()],
)

torch.cuda.reset_peak_memory_stats()
start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print("Training complete.")
print(f"Train runtime: {elapsed/60:.2f} minutes")
print(train_result)

In [ ]:
# ---- 8b) (Optional) Quick qualitative check on validation samples ----
# This cell does NOT change training; it just shows how the tuned model behaves
'# on a few held-out MedMCQA questions. Safe to skip if you like.',
from datasets import load_dataset as _load_dataset_eval

try:
    val_raw = _load_dataset_eval(DATASET_NAME, split="validation[:10]")
    print(f"Loaded {len(val_raw)} validation examples for a quick check.")
except Exception as e:
    print("Could not load validation split:", e)
    val_raw = None

if val_raw is not None:
    for i in range(len(val_raw)):
        ex = val_raw[i]
        q = ex.get("question", "").strip()
        print("=" * 80)
        print(f"Validation example {i+1}")
        print("Question:", q)
        print("Correct option index (cop):", ex.get("cop", "?"))
        print("Model answer (truncated):\n")
        ans = ask_medical(q, max_new_tokens=GEN_MAX_NEW_TOKENS)
        print(ans[:800])
        print()

In [ ]:
# ---- 9) Save adapter + tokenizer ----
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter/tokenizer saved to: {OUTPUT_DIR}")

if USE_DRIVE:
    model.save_pretrained(DRIVE_OUTPUT_DIR)
    tokenizer.save_pretrained(DRIVE_OUTPUT_DIR)
    print(f"Adapter/tokenizer also saved to Drive: {DRIVE_OUTPUT_DIR}")

In [ ]:
# ---- 10) Inference on new medical queries ----
FastLanguageModel.for_inference(model)

def ask_medical(query, max_new_tokens=None, temperature=None, top_p=None):
    """Generate an answer for a single medical question using the fine-tuned adapter.

    WARNING: This is a research model only. Do NOT use outputs for real patient care.
    """
    if max_new_tokens is None:
        max_new_tokens = GEN_MAX_NEW_TOKENS
    if temperature is None:
        temperature = GEN_TEMPERATURE
    if top_p is None:
        top_p = GEN_TOP_P

    prompt = (
        "Below is a medical question. Provide a clinically sound answer.\n\n"
        f"### Question:\n{query}\n\n"
        "### Context:\nNo additional context provided.\n\n"
        "### Answer:\n"
    )

    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            use_cache=True,
        )
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return decoded

def interactive_chat():
    """Simple loop to test the model on your own questions in Colab."""
    print("Type 'exit' to stop.\n")
    while True:
        q = input("Medical question> ").strip()
        if q.lower() in {"exit", "quit"}:
            break
        if not q:
            continue
        print("\nModel answer:\n")
        print(ask_medical(q))
        print("\n" + "=" * 80 + "\n")

test_queries = [
    "A 62-year-old with crushing chest pain and diaphoresis arrives in ER. What are immediate first-line management steps?",
    "How do you differentiate iron deficiency anemia from thalassemia trait using common lab findings?",
    "What is the preferred initial treatment approach for diabetic ketoacidosis in adults?",
]

for i, q in enumerate(test_queries, start=1):
    print("=" * 80)
    print(f"Test Query {i}: {q}")
    print("-" * 80)
    print(ask_medical(q))
    print()

## Medical safety disclaimer
- This notebook is for **research and education only**.
- The fine-tuned model is **not** approved for clinical decision making.
- Always consult licensed medical professionals and validated guidelines for patient care.

## Optional next steps
- Increase `MAX_TRAIN_SAMPLES` and `NUM_EPOCHS` for better specialization.
- Switch to another supported base model in `BASE_MODEL`.
- Add a validation split and compute task metrics.
- Merge adapter into base model for standalone deployment if needed.